# Method: k-Nearest Neighbours Wheat Segmentation

This notebook implements a knn classifier as another classical ML method for the wheat/soil segmentation. We reuse the same 29-feature pipeline from the random forest method (17 colour featrues + 12 multi-scale Gaussian features) to allow for a controlled comparison between the two classifiers.

## 1. Load and Prepare the Wheat Dataset

In [18]:
import os
import time 
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import matplotlib.pyplot as plt
import openpyxl

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, jaccard_score

plt.rcParams["figure.figsize"] = (10, 5)

In [5]:
# paths
PROJECT_ROOT = Path("../..").resolve()
DATA_DIR = PROJECT_ROOT / "EWS-Dataset"

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "validation"
TEST_DIR = DATA_DIR / "test"

XLSX_PATH = Path("training_results_history.xlsx")

print("train exists     :", TRAIN_DIR.exists())
print("validation exists:", VAL_DIR.exists())
print("test exists      :", TEST_DIR.exists())

train exists     : True
validation exists: True
test exists      : True


## Helper Functions
Duplicated from the Random Forest notebook. Feature pipeline is identical to experiment 2.

In [6]:
def get_image_paths(split_dir):
    split_dir = Path(split_dir)
    all_pngs = sorted(split_dir.glob("*.png"))
    image_paths = [p for p in all_pngs if not p.name.endswith("_mask.png")]
    return image_paths


def get_mask_path(image_path):
    image_path = Path(image_path)
    return image_path.with_name(image_path.stem + "_mask.png")


def load_rgb_image(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"could not load image: {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img


def load_mask(path):
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(f"could not load mask: {path}")
    mask = (mask > 127).astype(np.uint8)
    return mask


def rgb_to_hsi_features(rgb_img):
    rgb = rgb_img.astype(np.float32) / 255.0
    r = rgb[:, :, 0]
    g = rgb[:, :, 1]
    b = rgb[:, :, 2]

    intensity = (r + g + b) / 3.0
    min_rgb = np.minimum(np.minimum(r, g), b)

    saturation = np.zeros_like(intensity)
    valid = intensity > 1e-8
    saturation[valid] = 1.0 - (min_rgb[valid] / intensity[valid])

    num = 0.5 * ((r - g) + (r - b))
    den = np.sqrt((r - g) ** 2 + (r - b) * (g - b)) + 1e-8
    theta = np.arccos(np.clip(num / den, -1.0, 1.0))

    hue = np.where(b <= g, theta, 2 * np.pi - theta)
    hue = hue / (2 * np.pi)

    return hue, saturation, intensity


def compute_pixel_features(rgb_img):
    rgb = rgb_img.astype(np.float32) / 255.0
    r = rgb[:, :, 0]
    g = rgb[:, :, 1]
    b = rgb[:, :, 2]

    hsv = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2HSV).astype(np.float32)
    h_hsv = hsv[:, :, 0] / 179.0
    s_hsv = hsv[:, :, 1] / 255.0
    v_hsv = hsv[:, :, 2] / 255.0

    lab = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2LAB).astype(np.float32)
    l_lab = lab[:, :, 0] / 255.0
    a_lab = (lab[:, :, 1] - 128.0) / 127.0
    b_lab = (lab[:, :, 2] - 128.0) / 127.0

    ycrcb = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2YCrCb).astype(np.float32)
    y_ycc = ycrcb[:, :, 0] / 255.0
    cr_ycc = (ycrcb[:, :, 1] - 128.0) / 127.0
    cb_ycc = (ycrcb[:, :, 2] - 128.0) / 127.0

    h_hsi, s_hsi, i_hsi = rgb_to_hsi_features(rgb_img)

    exg = 2 * g - r - b
    ngrdi = (g - r) / (g + r + 1e-8)

    features = np.stack(
        [
            r, g, b,
            h_hsv, s_hsv, v_hsv,
            l_lab, a_lab, b_lab,
            y_ycc, cr_ycc, cb_ycc,
            h_hsi, s_hsi, i_hsi,
            exg, ngrdi,
        ],
        axis=-1,
    )

    h, w, c = features.shape
    return features.reshape(h * w, c)


def compute_multiscale_features(rgb_img, sigmas=(1, 2, 4, 8)):
    rgb = rgb_img.astype(np.float32) / 255.0
    r, g, b = rgb[:, :, 0], rgb[:, :, 1], rgb[:, :, 2]

    lab = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2LAB).astype(np.float32)
    a_lab = (lab[:, :, 1] - 128.0) / 127.0

    exg = 2 * g - r - b
    ngrdi = (g - r) / (g + r + 1e-8)   # eps avoids div-by-0 on saturated pixels

    feats = []
    for ch in [a_lab, exg, ngrdi]:
        for s in sigmas:
            feats.append(cv2.GaussianBlur(ch, (0, 0), sigmaX=s, sigmaY=s))

    stacked = np.stack(feats, axis=-1)
    h, w, c = stacked.shape
    return stacked.reshape(h * w, c)


def compute_pixel_features_v2(rgb_img):
    base = compute_pixel_features(rgb_img)
    ms = compute_multiscale_features(rgb_img)
    return np.hstack([base, ms])


def log_run(xlsx_path, run_name, model, hparams, precision, recall, f1, iou, train_time_s, eval_time_s, notes=""):
    wb = openpyxl.load_workbook(xlsx_path)
    ws = wb.active

    hparams_str = (
        ", ".join(f"{k}={v}" for k, v in hparams.items())
        if isinstance(hparams, dict) else str(hparams)
    )

    ws.append([
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        run_name,
        model,
        hparams_str,
        round(float(precision), 4),
        round(float(recall), 4),
        round(float(f1), 4),
        round(float(iou), 4),
        round(float(train_time_s), 2),
        round(float(eval_time_s), 2),
        notes
    ])
    wb.save(xlsx_path)
    print(f"logged: {run_name}")

In [12]:
def build_training_set(split_dir, pixels_per_image=350, max_images=None, random_state=42):
    rng = np.random.default_rng(random_state)
    image_paths = get_image_paths(split_dir)

    if max_images is not None:
        image_paths = image_paths[:max_images]

    x_list, y_list = [], []
    for image_path in image_paths:
        img = load_rgb_image(image_path)
        mask = load_mask(get_mask_path(image_path))

        x_img = compute_pixel_features_v2(img)
        y_img = mask.reshape(-1)

        n = min(pixels_per_image, len(y_img))
        idx = rng.choice(len(y_img), size=n, replace=False)

        x_list.append(x_img[idx])
        y_list.append(y_img[idx])

    return np.vstack(x_list), np.concatenate(y_list)

x_train, y_train = build_training_set(
    TRAIN_DIR,
    pixels_per_image=350,
    max_images=None,
    random_state=42,
)

print(f"x_train shape: {x_train.shape}")
print(f"plant pixels: {int(y_train.sum())}")
print(f"soil pixels : {int((y_train == 0).sum())}")

x_train shape: (49700, 29)
plant pixels: 39245
soil pixels : 10455


## Train the k-NN Model

In [13]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)

start_train = time.time()

knn = KNeighborsClassifier(
    n_neighbors=5,
    weights="uniform",
    algorithm="auto",
    n_jobs=-1,
)
knn.fit(x_train_scaled, y_train)

train_time = time.time() - start_train
print(f"fit finished in {train_time:.2f} seconds")

fit finished in 0.01 seconds


## Prediction and Evaluation Helpers

In [14]:
def predict_mask(model, scaler, rgb_img):
    x = compute_pixel_features_v2(rgb_img)
    x_scaled = scaler.transform(x)
    pred = model.predict(x_scaled)
    return pred.reshape(rgb_img.shape[:2]).astype(np.uint8)


def evaluate_split(split_dir, model, scaler, max_images=None):
    image_paths = get_image_paths(split_dir)
    if max_images is not None:
        image_paths = image_paths[:max_images]

    image_metrics, predictions, ground_truths, image_names = [], [], [], []

    start = time.time()
    for image_path in image_paths:
        img = load_rgb_image(image_path)
        gt = load_mask(get_mask_path(image_path))
        pred = predict_mask(model, scaler, img)

        image_metrics.append([
            precision_score(gt.reshape(-1), pred.reshape(-1), zero_division=0),
            recall_score(gt.reshape(-1), pred.reshape(-1), zero_division=0),
            f1_score(gt.reshape(-1), pred.reshape(-1), zero_division=0),
            jaccard_score(gt.reshape(-1), pred.reshape(-1), zero_division=0),
        ])
        predictions.append(pred)
        ground_truths.append(gt)
        image_names.append(image_path.name)
    test_time = time.time() - start

    image_metrics = np.array(image_metrics)
    summary = {
        "precision": float(image_metrics[:, 0].mean()),
        "recall": float(image_metrics[:, 1].mean()),
        "f1": float(image_metrics[:, 2].mean()),
        "iou": float(image_metrics[:, 3].mean()),
        "time_seconds": float(test_time),
        "num_images": len(image_paths),
    }
    return summary, image_names, predictions, ground_truths

## Validate on the validation split

In [15]:
val_summary, val_names, val_preds, val_gts = evaluate_split(VAL_DIR, knn, scaler)

print("validation summary (k=5 baseline)")
for k, v in val_summary.items():
    print(f"{k:>12}: {v:.4f}" if isinstance(v, float) else f"{k:>12}: {v}")

validation summary (k=5 baseline)
   precision: 0.9464
      recall: 0.8917
          f1: 0.9150
         iou: 0.8533
time_seconds: 145.6287
  num_images: 24


## Final evaluation on the test split

In [ ]:
# Final eval
# test_summary, test_names, test_preds, test_gts = evaluate_split(TEST_DIR, knn_best, scaler)

# print("test summary")
# for k, v in test_summary.items():
#     print(f"{k:>12}: {v:.4f}" if isinstance(v, float) else f"{k:>12}: {v}")

## 4. Experiment 1 - k sweep

In [19]:
configs = [
    dict(n_neighbors=k, weights="uniform", algorithm="auto")
    for k in [3, 5, 7, 15, 31]
]

print(f"total configs: {len(configs)}\n")

for i, cfg in enumerate(configs, 1):
    print(f"[{i}/{len(configs)}]")

    start = time.time()
    knn_cfg = KNeighborsClassifier(
        **cfg,
        n_jobs=-1,
    )
    knn_cfg.fit(x_train_scaled, y_train)
    train_time = time.time() - start

    val_summary, _, _, _ = evaluate_split(VAL_DIR, knn_cfg, scaler)

    run_name=f"sweep_k{cfg['n_neighbors']}"
    log_run(
        XLSX_PATH,
        run_name=run_name,
        model="kNN",
        hparams=cfg,
        precision=val_summary["precision"],
        recall=val_summary["recall"],
        f1=val_summary["f1"],
        iou=val_summary["iou"],
        train_time_s=train_time,
        eval_time_s=val_summary["time_seconds"],
        notes="val split; ~50k training pixels",
    )

    print(f"  train={train_time:.1f}s  val F1={val_summary['f1']:.4f}  IoU={val_summary['iou']:.4f}\n")

print("Sweep Complete")

total configs: 5

[1/5]
logged: sweep_k3
  train=0.0s  val F1=0.9134  IoU=0.8499

[2/5]
logged: sweep_k5
  train=0.0s  val F1=0.9150  IoU=0.8533

[3/5]
logged: sweep_k7
  train=0.0s  val F1=0.9149  IoU=0.8538

[4/5]
logged: sweep_k15
  train=0.0s  val F1=0.9116  IoU=0.8512

[5/5]
logged: sweep_k31
  train=0.0s  val F1=0.9054  IoU=0.8451

Sweep Complete
